In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import joblib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import (
    GridSearchCV,
    learning_curve,
    train_test_split
)

from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso


from catboost import CatBoostRegressor
from transformers.transformers import *
from transformers import *

ModuleNotFoundError: No module named 'catboost'

In [ ]:
data = pd.read_csv('data/laptop_data.csv')

X = data.drop(columns=['Price', 'Unnamed: 0'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [ ]:
feature_engineering = ColumnTransformer(
    transformers=[
    ('company', CompanyBinning(), ['Company']),
    ('typename', TypeNameEncoder(), ['TypeName']),
    ('inches', InchesBinning(), ['Inches']),
    ('screen', ScreenResolutionEncoder(), ['ScreenResolution']),
    ('cpu', CpuEncoder(), ['Cpu']),
    ('ram', RamBinning(), ['Ram']),
    ('memory', MemoryEncoder(), ['Memory']),
    ('gpu', GpuEncoder(), ['Gpu']),
    ('opsys', OpSysEncoder(), ['OpSys']),
    ('weight', WeightExtract(), ['Weight'])
], remainder='drop')

In [ ]:
catboost_fs = CatBoostRegressor(
    iterations=800,
    learning_rate=0.03,
    depth=3,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=0
)
feature_selector = SelectFromModel(catboost_fs, threshold='median')

In [ ]:
preprocess = Pipeline([
    ('feature_eng', feature_engineering),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipeline_fs = Pipeline([
    ('preprocess', preprocess),
    ('feature_selection', feature_selector),
    ('model', Lasso(max_iter=50000))
])

param_grid = {
    'model__alpha': [1, 20, 50, 100],
}

grid = GridSearchCV(
    pipeline_fs,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=1
)

grid.fit(X_train, y_train)

In [ ]:
print("Best R²:", grid.best_score_)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    grid.best_estimator_,
    X_train,
    y_train,
    cv=5,
    scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)

val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(train_sizes, train_mean, label='Train R²')
plt.fill_between(
    train_sizes,
    train_mean - train_std,
    train_mean + train_std,
    alpha=0.2
)

plt.plot(train_sizes, val_mean, label='Validation R²')
plt.fill_between(
    train_sizes,
    val_mean - val_std,
    val_mean + val_std,
    alpha=0.2
)

plt.xlabel("Training Size")
plt.ylabel("R² Score")
plt.title("Learning Curve (R²)")
plt.legend()
plt.grid(True)
output_path ="outputs/figures/learning_curve.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
joblib.dump(grid.best_estimator_, "outputs/models/laptop_price_lasso_model.pkl")
print("Model saved as laptop_price_lasso_model.pkl")